# 02 — Preprocessing
Notebook ini menyiapkan data untuk tahap modeling, berdasarkan temuan EDA (`01_EDA`):

| Temuan EDA | Keputusan preprocessing |
|---|---|
| Tidak ada missing value & duplikat | Tidak perlu imputasi / penghapusan baris |
| `Unnamed: 0` = indeks, `flight` = kode penerbangan (mirip ID) | Kedua kolom dibuang |
| Harga naik konsisten: zero < one < two_or_more transit | `stops` → ordinal 0 / 1 / 2 |
| `class` hanya 2 nilai | `class` → `is_business` (0/1) |
| Hubungan `days_left`–harga **tidak linear**: datar di ≥16 hari, naik tajam mendekati keberangkatan | Dibuat fitur kelompok `days_group` (16+, 11–15, 4–10, 1–3) untuk MLR; `days_left` asli tetap disimpan untuk Decision Tree & KNN |
| Harga miring & bimodal; `log(harga)` per kelas mendekati normal | Target = `log_price` |
| Data asli berurutan per rute/kelas/hari | Train/test split **diacak** (80/20) |


In [1]:
# Sel 1 — muat data
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = '/content/drive/MyDrive/Project Data mining/Dataset'
df = pd.read_csv(f'{BASE}/Clean_Dataset.csv')
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(300153, 12)


## 1. Buang kolom yang tidak dipakai

In [2]:
# Sel 2
df = df.drop(columns=['Unnamed: 0', 'flight'])
print(df.columns.tolist())

['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left', 'price']


## 2. Encoding kolom ordinal & biner

In [3]:
# Sel 3
df['stops'] = df['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})
df['is_business'] = (df['class'] == 'Business').astype(int)
df = df.drop(columns=['class'])

# cek: tidak boleh ada NaN akibat mapping yang meleset
print(df[['stops', 'is_business']].isnull().sum())
print(df['stops'].value_counts().sort_index())

stops          0
is_business    0
dtype: int64
stops
0     36004
1    250863
2     13286
Name: count, dtype: int64


## 3. Feature engineering: kelompok waktu pemesanan
Batas kelompok diambil dari grafik Sel 8 di EDA. Kategori **16+** dijadikan baseline (harga "normal"), sehingga koefisien kelompok lain di MLR terbaca sebagai *berapa persen lebih mahal* jika membeli mendadak.

In [4]:
# Sel 4
bins   = [0, 3, 10, 15, 49]
labels = ['1-3', '4-10', '11-15', '16+']
df['days_group'] = pd.cut(df['days_left'], bins=bins, labels=labels)

# urutan kategori: baseline (16+) diletakkan pertama agar yang dibuang oleh drop_first
df['days_group'] = pd.Categorical(df['days_group'], categories=['16+', '11-15', '4-10', '1-3'])

print(df['days_group'].value_counts())
print("\nRata-rata harga per kelompok & kelas:")
print(df.groupby(['days_group', 'is_business'], observed=True)['price'].mean().unstack().round(0))

days_group
16+      218895
4-10      39166
11-15     31891
1-3       10201
Name: count, dtype: int64

Rata-rata harga per kelompok & kelas:
is_business        0        1
days_group                   
16+           5135.0  51854.0
11-15         9220.0  52162.0
4-10         10759.0  54368.0
1-3          13808.0  60795.0


## 4. One-hot encoding kolom kategorikal
`drop_first=True` membuang satu kategori per kolom (menjadi baseline) untuk menghindari multikolinearitas sempurna (*dummy variable trap*) di regresi linear.

In [5]:
# Sel 5
kategorikal = ['airline', 'source_city', 'destination_city',
               'departure_time', 'arrival_time', 'days_group']
df_enc = pd.get_dummies(df, columns=kategorikal, drop_first=True, dtype=int)

print("Ukuran setelah encoding:", df_enc.shape)
print("\nKategori baseline (yang dibuang):")
for col in kategorikal:
    asli = df[col].cat.categories[0] if hasattr(df[col], 'cat') else sorted(df[col].unique())[0]
    print(f"  {col:17s}: {asli}")

Ukuran setelah encoding: (300153, 33)

Kategori baseline (yang dibuang):
  airline          : AirAsia
  source_city      : Bangalore
  destination_city : Bangalore
  departure_time   : Afternoon
  arrival_time     : Afternoon
  days_group       : 16+


## 5. Transformasi target

In [6]:
# Sel 6
df_enc['log_price'] = np.log(df_enc['price'])
print(df_enc[['price', 'log_price']].describe().round(3))

            price   log_price
count  300153.000  300153.000
mean    20889.661       9.331
std     22697.767       1.113
min      1105.000       7.008
25%      4783.000       8.473
50%      7425.000       8.913
75%     42521.000      10.658
max    123071.000      11.721


## 6. Train/test split (80/20, diacak)
Normalisasi (scaling) **tidak** dilakukan di sini. Scaling hanya dibutuhkan KNN dan harus di-*fit* pada data train saja, jadi akan dilakukan di notebook modeling agar tidak terjadi kebocoran data (data leakage).

In [7]:
# Sel 7
train, test = train_test_split(df_enc, test_size=0.2, random_state=42, shuffle=True)
print("Train:", train.shape, "| Test:", test.shape)

# cek proporsi kelas tetap mirip di train & test
print("Proporsi Business — train: %.3f | test: %.3f" % (train['is_business'].mean(), test['is_business'].mean()))

Train: (240122, 34) | Test: (60031, 34)
Proporsi Business — train: 0.311 | test: 0.312


## 7. Simpan hasil ke `Dataset/Processed`

In [8]:
# Sel 8
out = f'{BASE}/Processed'
os.makedirs(out, exist_ok=True)
train.to_csv(f'{out}/train.csv', index=False)
test.to_csv(f'{out}/test.csv', index=False)
print("Tersimpan di:", out)
print(train.columns.tolist())

Tersimpan di: /content/drive/MyDrive/Project Data mining/Dataset/Processed
['stops', 'duration', 'days_left', 'price', 'is_business', 'airline_Air_India', 'airline_GO_FIRST', 'airline_Indigo', 'airline_SpiceJet', 'airline_Vistara', 'source_city_Chennai', 'source_city_Delhi', 'source_city_Hyderabad', 'source_city_Kolkata', 'source_city_Mumbai', 'destination_city_Chennai', 'destination_city_Delhi', 'destination_city_Hyderabad', 'destination_city_Kolkata', 'destination_city_Mumbai', 'departure_time_Early_Morning', 'departure_time_Evening', 'departure_time_Late_Night', 'departure_time_Morning', 'departure_time_Night', 'arrival_time_Early_Morning', 'arrival_time_Evening', 'arrival_time_Late_Night', 'arrival_time_Morning', 'arrival_time_Night', 'days_group_11-15', 'days_group_4-10', 'days_group_1-3', 'log_price']
